# FLAG 2027 - Notebook 25: a public cross-modal head (MSM, Moscati et al. ICASSP 2026) as a frozen member

MSM is the best published unseen-unheard face-voice verification result on VoxCeleb1 (EER 22.9). Its released checkpoints (github hcai-mms/MSM-face-voice, 5.8 MB) are a tiny head on frozen **FaceNet 512-d** (facenet-pytorch, VGGFace2) and **ECAPA-TDNN 192-d** features, trained on VoxCeleb1 only (no MAV-Celeb, no Bengali). We use it strictly frozen, two ways:

- **M1 features**: its 900-d face and voice projections become extra blocks for our own head.
- **M2 fusion**: its pair score (negative distance of the two projections) is added to our score with a FIXED weight computed from TRAINING pairs (std matching), i.e. no test statistics - allowed.

Unknown until we run it: whether MSM's ECAPA is the same extractor as the organizers' (ours reproduces theirs at cosine 0.9999). Cell 3 therefore measures MSM's own EER on challenge TRAINING pairs; if it is near 50, the voice input is mismatched and we stop.

Steps: face lists (alignment check) -> FaceNet features -> MSM checkpoints + sanity -> CV: M1, M2 -> ZIP of the best (no VoxCeleb). About 1.5-2 h. Paste the **SUMMARY**.

In [ ]:
# Cell 1 [v2 29-Oct] - Drive, GPU, packages (facenet-pytorch pins an old Pillow that fails to build on Colab -> install without its dependency pins;
#                       torch, torchvision, numpy, Pillow and requests are already present)
from google.colab import drive
drive.mount('/content/drive')
!pip -q install --no-deps facenet-pytorch
import torch, shutil
from facenet_pytorch import InceptionResnetV1, fixed_image_standardization   # fails loudly here if the install did not work
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('[v2 29-Oct] facenet-pytorch OK | Device:', dev, '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - FACE file lists in our row order (same code as NB23) + FaceNet 512-d extraction (fixed_image_standardization, 160 px)
import os, re, sys, json, time, zipfile, copy
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results'); CACHE = os.path.join(PROJECT, 'models_cache')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
MAV_ZIPS, MAV_LISTS = os.path.join(BASE, 'dataset'), os.path.join(BASE, 'face_voice_association_splits')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
IMG = ('.jpg', '.jpeg', '.png')
FULL = '/content/flag_faces'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(FULL, members=[m for m in zf.namelist() if m.lower().endswith(IMG + ('.txt', '.csv'))])
        open(os.path.join(FULL, z + '.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
def face_token(tokens):
    t = [x for x in tokens if x.lower().endswith(IMG)]; assert len(t) == 1, tokens; return t[0]
SPLIT_NAMES = ['train', 'no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
FILES = {}
lp = find_one('train_English.txt'); FILES['train'] = [resolve(face_token(l.split()), lp) for l in open(lp) if l.strip()]
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); lp = find_one('/' + track + '/', lang_ + '_test.txt')
    FILES[s] = [resolve(face_token(l.split()), lp) for l in open(lp) if l.strip()]
assert all(None not in v for v in FILES.values()), 'some face files missing - tell Claude'
def zip_index(zf):
    m = {}
    for n in zf.namelist():
        for key in ('voices/', 'faces/'):
            i = n.find(key)
            if i >= 0 and not n.endswith('/'):
                m[n[i:]] = n; break
    return m
MAVD = '/content/mav'; mav_paths, mspk = [], []
for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
    flag = os.path.join(MAVD, ver + '.faces.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf)
    for lf in lists:
        for l in open(os.path.join(MAV_LISTS, ver, lf)):
            t = l.split()
            if len(t) < 2:
                continue
            vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
            if vm is None or fm is None:
                continue
            if need:
                zf.extract(fm, os.path.join(MAVD, ver))
            mav_paths.append(os.path.join(MAVD, ver, fm)); mspk.append(ver + '_' + t[0].split('/')[0])
    zf.close()
    if need:
        open(flag, 'w').close()
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz')))
assert list(MAV['spk']) == mspk, 'MAV v1/v3 row order differs - tell Claude'
FILES['mav'] = mav_paths
EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
MAX_PER_ID = 120
VPAT = re.compile(r'voices/(id\d+)/([^/]+)/([^/]+)/([^/]+\.wav)$'); FPAT = re.compile(r'faces/(id\d+)/([^/]+)/([^/]+)/([^/]+\.jpg)$')
v1_used = set()
for lf in ['English_train.txt', 'Urdu_train.txt']:
    for l in open(os.path.join(MAV_LISTS, 'v1', lf)):
        t = l.split()
        if t:
            v1_used.add(t[0].split('/')[0])
rows = []
for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
    D = '/content/mavx/' + ver; flag = os.path.join(D, 'faces.done'); need = not os.path.exists(flag)
    zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
    for n in zf.namelist():
        mv = VPAT.search(n)
        if mv:
            voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
        mf = FPAT.search(n)
        if mf:
            faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
    ids_ = sorted(i for i in voices if i not in skip_ids); rng = np.random.default_rng(0)
    for pid in ids_:
        vs = voices[pid]; vs = [vs[j] for j in rng.permutation(len(vs))[:MAX_PER_ID]]
        for lang_, vid, vn in vs:
            fr = faces.get((pid, lang_, vid)) or [f for (p, l, v), fl in faces.items() if p == pid for f in fl]
            if not fr:
                continue
            fn = fr[rng.integers(len(fr))]
            if need:
                zf.extract(fn, D)
            rows.append((ver + '_' + pid, os.path.join(D, fn)))
    zf.close()
    if need:
        open(flag, 'w').close()
spk_all = np.array([r[0] for r in rows]); keep = np.ones(len(rows), bool)
if len(rows) != len(EXTRA['spk']):
    j = 0; keep[:] = False
    for i in range(len(rows)):
        if j < len(EXTRA['spk']) and spk_all[i] == EXTRA['spk'][j]:
            keep[i] = True; j += 1
    assert j == len(EXTRA['spk']), 'could not align MAV extra rows - tell Claude'
FILES['extra'] = [r[1] for r, k in zip(rows, keep) if k]
from transformers import CLIPModel, CLIPProcessor
_cm = CLIPModel.from_pretrained('openai/clip-vit-large-patch14').half().to(dev).eval(); _cp = CLIPProcessor.from_pretrained('openai/clip-vit-large-patch14')
def clip_ref(paths):
    x = _cp(images=[Image.open(p).convert('RGB') for p in paths], return_tensors='pt')['pixel_values'].to(dev, torch.float16)
    with torch.no_grad():
        return _cm.visual_projection(_cm.vision_model(pixel_values=x).pooler_output).float().cpu().numpy()
CLIPF = np.load(os.path.join(FEATDIR, 'clip.npz'))
rng = np.random.default_rng(1)
for k, saved in {'train': CLIPF['train'], 'mav': MAV['clip'], 'extra': EXTRA['clip']}.items():
    idx = rng.choice(len(saved), 100, replace=False); new = clip_ref([FILES[k][i] for i in idx]); old = saved[idx]
    cos = (new * old).sum(1) / (np.linalg.norm(new, axis=1) * np.linalg.norm(old, axis=1) + 1e-9)
    note('face alignment %-6s median cosine %.4f' % (k, np.median(cos))); assert np.median(cos) > 0.98, 'face rows misaligned - tell Claude'
del _cm; torch.cuda.empty_cache()
ORDER = SPLIT_NAMES + ['mav', 'extra']; allp = [p for s in ORDER for p in FILES[s]]
def split_back(emb):
    out, i = {}, 0
    for s in ORDER:
        n = len(FILES[s]); out[s] = emb[i:i + n].astype(np.float32); i += n
    return out
FN_PATH = os.path.join(FEATDIR, 'facenet.npz')
if not os.path.exists(FN_PATH):
    from facenet_pytorch import InceptionResnetV1, fixed_image_standardization
    net = InceptionResnetV1(pretrained='vggface2').to(dev).eval(); out = []; t0 = time.time()
    def prep(p):
        im = Image.open(p).convert('RGB').resize((160, 160), Image.BILINEAR)
        return fixed_image_standardization(torch.from_numpy(np.asarray(im, np.float32)).permute(2, 0, 1))
    for i in tqdm(range(0, len(allp), 128), desc='facenet'):
        x = torch.stack([prep(p) for p in allp[i:i + 128]]).to(dev)
        with torch.no_grad():
            out.append(net(x).cpu().numpy())
    np.savez(FN_PATH, **split_back(np.concatenate(out))); note('facenet 512-d extracted for %d faces in %.1f min' % (len(allp), (time.time() - t0) / 60)); del net; torch.cuda.empty_cache()
FACENET = dict(np.load(FN_PATH))

In [ ]:
# Cell 3 [v3 29-Oct] - MSM checkpoints: BatchNorm-statistics match test per branch (does our input look like their training input?),
#                       fallbacks (MTCNN-aligned FaceNet for faces; TaoRuijie ECAPA for voices), then the cross-modal gate on TRAIN pairs
import requests, subprocess
import torch.nn.functional as Fn
import librosa
from sklearn.metrics import roc_curve
MSMDIR = os.path.join(CACHE, 'msm'); os.makedirs(MSMDIR, exist_ok=True)
CKPTS = ['MSM_checkpoint_0016.pth.tar', 'MSM_checkpoint_0013.pth.tar', 'MSM_checkpoint_0010.pth.tar', 'MSM_alpha_checkpoint_0005.pth.tar']
for c in CKPTS:
    p = os.path.join(MSMDIR, c)
    if not os.path.exists(p):
        r = requests.get('https://raw.githubusercontent.com/hcai-mms/MSM-face-voice/master/best_models/' + c, timeout=120); r.raise_for_status(); open(p, 'wb').write(r.content)
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
ECAPA = {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}
for s in SPLIT_NAMES[1:]:
    track, lang_ = s.rsplit('_', 1); ECAPA[s] = read_feat(find_one('/' + track + '/', lang_ + '_test_voices.csv'), 192)[0]
class Branch(torch.nn.Module):
    def __init__(self, sd, prefix):
        super().__init__()
        keys = [k for k in sd if k.startswith(prefix)]
        W = [k for k in keys if sd[k].ndim == 2]; assert len(W) == 1, (prefix, keys); W = W[0]; base = W[:-len('weight')]
        self.lin = torch.nn.Linear(sd[W].shape[1], sd[W].shape[0]); self.lin.weight.data = sd[W].float(); self.lin.bias.data = sd[base + 'bias'].float()
        bn = sorted({k.rsplit('.', 1)[0] for k in keys if k.endswith('running_mean')}); assert len(bn) == 1, (prefix, keys); bn = bn[0]
        self.bn = torch.nn.BatchNorm1d(sd[W].shape[0]); self.bn.weight.data = sd[bn + '.weight'].float(); self.bn.bias.data = sd[bn + '.bias'].float()
        self.bn.running_mean = sd[bn + '.running_mean'].float(); self.bn.running_var = sd[bn + '.running_var'].float()
    def forward(self, x):
        return Fn.normalize(torch.relu(self.bn(self.lin(x))), dim=1)
def load_msm(c):
    ck = torch.load(os.path.join(MSMDIR, c), map_location='cpu', weights_only=False); sd = ck.get('state_dict', ck)
    sd = {k[len('module.'):] if k.startswith('module.') else k: v for k, v in sd.items() if torch.is_tensor(v)}
    fpre = [k for k in sd if 'face' in k and k.endswith('weight') and sd[k].ndim == 2][0].split('.')[0]
    vpre = [k for k in sd if 'voice' in k and k.endswith('weight') and sd[k].ndim == 2][0].split('.')[0]
    return Branch(sd, fpre + '.').to(dev).eval(), Branch(sd, vpre + '.').to(dev).eval()
@torch.no_grad()
def project(branch, X, bs=4096):
    return np.concatenate([branch(torch.tensor(X[i:i + bs], device=dev, dtype=torch.float32)).cpu().numpy() for i in range(0, len(X), bs)])
@torch.no_grad()
def bn_match(branch, X):
    """pre-BN activations of OUR data vs the BN running statistics stored in the checkpoint: corr of per-unit means, median variance ratio"""
    z = branch.lin(torch.tensor(X[:6000], device=dev, dtype=torch.float32)); mu, var = z.mean(0).cpu().numpy(), z.var(0).cpu().numpy()
    rm, rv = branch.bn.running_mean.cpu().numpy(), branch.bn.running_var.cpu().numpy()
    return float(np.corrcoef(mu, rm)[0, 1]), float(np.median(var / (rv + 1e-9)))
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
rng = np.random.default_rng(3); spk_ = np.unique(y); rows_ = {s: np.where(y == s)[0] for s in spk_}
PV, PF, PL = [], [], []
for _ in range(4000):
    s = rng.choice(spk_); PV.append(rng.choice(rows_[s])); PF.append(rng.choice(rows_[s])); PL.append(1)
    s, t = rng.choice(spk_, 2, replace=False); PV.append(rng.choice(rows_[s])); PF.append(rng.choice(rows_[t])); PL.append(0)
PV, PF, PL = np.array(PV), np.array(PF), np.array(PL)
fb0, vb0 = load_msm(CKPTS[0])
# ---------- face candidates
def l2d(D):
    return {s: D[s] / (np.linalg.norm(D[s], axis=1, keepdims=True) + 1e-9) for s in ORDER}
FACE_SETS = {'resize': FACENET, 'resize_l2': l2d(FACENET)}
def extract_mtcnn():
    p = os.path.join(FEATDIR, 'facenet_mtcnn.npz')
    if not os.path.exists(p):
        from facenet_pytorch import MTCNN
        mt = MTCNN(image_size=160, margin=0, post_process=True, select_largest=True, device=dev); net = InceptionResnetV1(pretrained='vggface2').to(dev).eval()
        out, miss, t0 = [], 0, time.time()
        for i in tqdm(range(0, len(allp), 64), desc='mtcnn+facenet'):
            xs = []
            for q in allp[i:i + 64]:
                im = Image.open(q).convert('RGB'); x = mt(im)
                if x is None:
                    miss += 1; x = fixed_image_standardization(torch.from_numpy(np.asarray(im.resize((160, 160), Image.BILINEAR), np.float32)).permute(2, 0, 1))
                xs.append(x)
            with torch.no_grad():
                out.append(net(torch.stack(xs).to(dev)).cpu().numpy())
        np.savez(p, **split_back(np.concatenate(out))); note('MTCNN-aligned FaceNet extracted (%d/%d faces without detection -> resized) in %.0f min' % (miss, len(allp), (time.time() - t0) / 60)); del net, mt; torch.cuda.empty_cache()
    return dict(np.load(p))
# ---------- voice candidates
VOICE_SETS = {'ours': ECAPA, 'ours_l2': l2d(ECAPA)}
def extract_tao():
    p = os.path.join(FEATDIR, 'ecapa_tao.npz')
    if not os.path.exists(p):
        note('extracting TaoRuijie ECAPA-TDNN (exps/pretrain.model) for all rows')
        if not os.path.exists('/content/tao_ecapa'):
            subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/TaoRuijie/ECAPA-TDNN.git', '/content/tao_ecapa'], check=True)
        sys.path.insert(0, '/content/tao_ecapa'); import importlib; tao_model = importlib.import_module('model')
        net = tao_model.ECAPA_TDNN(C=1024)
        sd = torch.load('/content/tao_ecapa/exps/pretrain.model', map_location='cpu', weights_only=False)
        sd = {k[len('speaker_encoder.'):]: v for k, v in sd.items() if k.startswith('speaker_encoder.')}
        missing, unexpected = net.load_state_dict(sd, strict=False); assert not missing, missing[:5]; net = net.to(dev).eval()
        VFULL = '/content/flag_data'; os.makedirs(VFULL, exist_ok=True)
        for z in ['train_set.zip', 'dev_set.zip']:
            if not os.path.exists(os.path.join(VFULL, z + '.done')):
                with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
                    zf.extractall(VFULL, members=[m for m in zf.namelist() if m.endswith(('.wav', '.txt', '.csv'))])
                open(os.path.join(VFULL, z + '.done'), 'w').close()
        VTXT = [os.path.join(r, f) for r, _, fs in os.walk(VFULL) for f in fs if f.endswith('.txt')]
        def vfind(*parts):
            hits = [q for q in VTXT if all(x in q for x in parts)]; assert len(hits) == 1, (parts, hits); return hits[0]
        def vresolve(rel, lp):
            q = os.path.dirname(lp)
            while q.startswith(VFULL):
                c = os.path.join(q, rel)
                if os.path.exists(c):
                    return c
                q = os.path.dirname(q)
        def wav_token(tokens):
            t = [x for x in tokens if x.lower().endswith('.wav')]; assert len(t) == 1, tokens; return t[0]
        VF = {}
        lp = vfind('train_English.txt'); VF['train'] = [vresolve(wav_token(l.split()), lp) for l in open(lp) if l.strip()]
        for s in SPLIT_NAMES[1:]:
            track, lang_ = s.rsplit('_', 1); lp = vfind('/' + track + '/', lang_ + '_test.txt'); VF[s] = [vresolve(wav_token(l.split()), lp) for l in open(lp) if l.strip()]
        vm_paths = []
        for ver, zname, lists in [('v1', 'mavceleb_v1_train.zip', ['English_train.txt', 'Urdu_train.txt']), ('v3', 'mavceleb_v3_tran.zip', ['English_train.txt', 'German_train.txt'])]:
            zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); zi = zip_index(zf); flag = os.path.join(MAVD, ver + '.voices.done'); need = not os.path.exists(flag)
            for lf in lists:
                for l in open(os.path.join(MAV_LISTS, ver, lf)):
                    t = l.split()
                    if len(t) < 2:
                        continue
                    vm, fm = zi.get('voices/' + t[0]), zi.get('faces/' + t[1])
                    if vm is None or fm is None:
                        continue
                    if need:
                        zf.extract(vm, os.path.join(MAVD, ver))
                    vm_paths.append(os.path.join(MAVD, ver, vm))
            zf.close()
            if need:
                open(flag, 'w').close()
        assert len(vm_paths) == len(FILES['mav']); VF['mav'] = vm_paths
        xv_paths = []
        for ver, zname, skip_ids in [('v2', 'mavceleb_v2_complete.zip', set()), ('v1', 'mavceleb_v1_complete.zip', v1_used)]:
            D = '/content/mavx/' + ver; flag = os.path.join(D, 'voices.done'); need = not os.path.exists(flag)
            zf = zipfile.ZipFile(os.path.join(MAV_ZIPS, zname)); voices, faces = {}, {}
            for n in zf.namelist():
                mv = VPAT.search(n)
                if mv:
                    voices.setdefault(mv.group(1), []).append((mv.group(2), mv.group(3), n)); continue
                mf = FPAT.search(n)
                if mf:
                    faces.setdefault((mf.group(1), mf.group(2), mf.group(3)), []).append(n)
            ids_ = sorted(i for i in voices if i not in skip_ids); rng2 = np.random.default_rng(0)
            for pid in ids_:
                vs = voices[pid]; vs = [vs[j] for j in rng2.permutation(len(vs))[:MAX_PER_ID]]
                for lang_, vid, vn in vs:
                    fr = faces.get((pid, lang_, vid)) or [f for (pp, l, v), fl in faces.items() if pp == pid for f in fl]
                    if not fr:
                        continue
                    _ = fr[rng2.integers(len(fr))]
                    if need:
                        zf.extract(vn, D)
                    xv_paths.append(os.path.join(D, vn))
            zf.close()
            if need:
                open(flag, 'w').close()
        VF['extra'] = [q for q, k in zip(xv_paths, keep) if k]; assert len(VF['extra']) == len(FILES['extra'])
        @torch.no_grad()
        def tao_embed(paths, max_sec=20):
            out = []
            for q in tqdm(paths, desc='tao ecapa'):
                w = librosa.load(q, sr=16000, mono=True)[0][:16000 * max_sec]
                out.append(net(torch.from_numpy(w).unsqueeze(0).to(dev), aug=False).squeeze(0).float().cpu().numpy())
            return np.stack(out)
        t0 = time.time(); TAOF = {s: tao_embed(VF[s]).astype(np.float32) for s in ORDER}; np.savez(p, **TAOF)
        note('TaoRuijie ECAPA extracted for all rows in %.0f min' % ((time.time() - t0) / 60)); del net; torch.cuda.empty_cache()
    return dict(np.load(p))
# ---------- 1. BN match of the current inputs
rep = lambda name, br, D: note('BN match %-10s corr(means) %.3f | var ratio %.2f   (match: corr > 0.8, ratio 0.5-2)' % (name, *bn_match(br, D['train'])))
for n_, D in FACE_SETS.items():
    rep('face/' + n_, fb0, D)
for n_, D in VOICE_SETS.items():
    rep('voice/' + n_, vb0, D)
best_face = max(FACE_SETS, key=lambda n_: bn_match(fb0, FACE_SETS[n_]['train'])[0]); best_voice = max(VOICE_SETS, key=lambda n_: bn_match(vb0, VOICE_SETS[n_]['train'])[0])
# ---------- 2. fallbacks only where the match is poor
if bn_match(fb0, FACE_SETS[best_face]['train'])[0] < 0.8:
    M = extract_mtcnn(); FACE_SETS['mtcnn'] = M; FACE_SETS['mtcnn_l2'] = l2d(M)
    for n_ in ['mtcnn', 'mtcnn_l2']:
        rep('face/' + n_, fb0, FACE_SETS[n_])
if bn_match(vb0, VOICE_SETS[best_voice]['train'])[0] < 0.8:
    T = extract_tao(); VOICE_SETS['tao'] = T; VOICE_SETS['tao_l2'] = l2d(T)
    for n_ in ['tao', 'tao_l2']:
        rep('voice/' + n_, vb0, VOICE_SETS[n_])
# ---------- 3. cross-modal EER on TRAIN pairs for the best-matching inputs, all checkpoints; gate
best_face = max(FACE_SETS, key=lambda n_: bn_match(fb0, FACE_SETS[n_]['train'])[0]); best_voice = max(VOICE_SETS, key=lambda n_: bn_match(vb0, VOICE_SETS[n_]['train'])[0])
note('best-matching inputs: face=%s, voice=%s' % (best_face, best_voice))
best = None
for c in CKPTS:
    fb, vb = load_msm(c); pf, pv = project(fb, FACE_SETS[best_face]['train']), project(vb, VOICE_SETS[best_voice]['train'])
    e = eer(PL, -np.linalg.norm(pf[PF] - pv[PV], axis=1)); note('%s: cross-modal EER on 8,000 challenge TRAIN pairs = %.2f' % (c.replace('.pth.tar', ''), e))
    if best is None or e < best[0]:
        best = (e, c, fb, vb)
assert best[0] < 45, 'MSM is near chance even with matched inputs -> unusable; STOP and tell Claude'
MSM_EER, MSM_CK, FB, VB = best; FS, VS = FACE_SETS[best_face], VOICE_SETS[best_voice]
note('MSM member = %s (face=%s, voice=%s; train-pair EER %.2f)' % (MSM_CK, best_face, best_voice, MSM_EER))
MSMF = {s: project(FB, FS[s]) for s in ORDER}; MSMV = {s: project(VB, VS[s]) for s in ORDER}

In [ ]:
# Cell 4 - CV (challenge + all MAV, CLIP | R4, prototypes; 5-fold x 3 seeds): M1 = MSM projections as extra feature blocks; M2 = fixed-weight score fusion
import torch.nn as nn
FEAT = {'ecapa': ECAPA, 'msm_f': MSMF, 'msm_v': MSMV}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
N_CH = len(y); ext_spk = list(MAV['spk']) + list(EXTRA['spk'])
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
MSM_TRAIN = -np.linalg.norm(0, axis=0) if False else None
def msm_score(Vi, Fi, split_v='train', split_f='train'):
    return -np.linalg.norm(MSMF[split_f][Fi] - MSMV[split_v][Vi], axis=1)
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA = []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)], make_pairs(tr_idx, seed=500 + k)))
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class HBranch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = HBranch(d_face, d), HBranch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_combo(F_raw, V_raw, tr_idx, vsets=None, seed=1, max_epochs=MAX_EP):
    torch.manual_seed(seed); rng = np.random.default_rng(seed); all_tr = np.concatenate([tr_idx, EXT])
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            loss = Fn.cross_entropy(logits, ytr[j]) + ALPHA * opl_loss(fused, ytr[j]); opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']
COMBOS = {'M0_ref': (['clip'], R4, 0.0), 'M1_msmfeat': (['clip', 'msm_f'], R4 + ['msm_v'], 0.0), 'M2_fuse1.0': (['clip'], R4, 1.0), 'M2_fuse0.5': (['clip'], R4, 0.5)}
CSV = os.path.join(RESDIR, 'nb25_cv.csv'); done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold']); t0 = time.time()
# M0 and both M2 variants share the same trained head: train once per (seed, fold) and record all three rows
for seed in SEEDS:
    for k in range(K):
        tr_idx, vsets, trpairs = FOLD_DATA[k]
        for name, (fn_, vn_, lam) in COMBOS.items():
            if lam > 0 or ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            F_raw, V_raw = build_train(fn_), build_train(vn_)
            model, best_ep, XF, XV, _ = train_combo(F_raw, V_raw, tr_idx, vsets, seed=seed)
            ours = [same_score(model, XF[Fi], XV[Vi]) for Vi, Fi, L in vsets]
            rows = [dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=eer(vsets[0][2], ours[0]), gender=eer(vsets[1][2], ours[1]))]
            if name == 'M0_ref':
                # fixed fusion weight from TRAINING pairs of this fold: match the two score scales, then add MSM with weight lam
                Vt, Ft, _ = trpairs; w = np.std(same_score(model, XF[Ft], XV[Vt])) / (np.std(msm_score(Vt, Ft)) + 1e-9)
                for fname, (_, _, lam) in COMBOS.items():
                    if lam > 0 and not ((done.combo == fname) & (done.seed == seed) & (done.fold == k)).any():
                        fused = [ours[i] + lam * w * msm_score(vsets[i][0], vsets[i][1]) for i in range(2)]
                        rows.append(dict(combo=fname, seed=seed, fold=k, best_ep=best_ep, std=eer(vsets[0][2], fused[0]), gender=eer(vsets[1][2], fused[1])))
            for r in rows:
                row = pd.DataFrame([r]); row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False); done = pd.concat([done, row])
                print('%-12s seed %d fold %d | ep %3d | std %.2f | gender %.2f | %.0f min' % (r['combo'], seed, k, r['best_ep'], r['std'], r['gender'], (time.time() - t0) / 60))
R = pd.read_csv(CSV); R['overall'] = (R['std'] + R['gender']) / 2
tab = R.groupby(['combo', 'seed'])[['std', 'gender', 'overall']].mean().groupby('combo').mean().round(2)
note('MSM AS A FROZEN MEMBER (challenge + all MAV, CLIP | R4, prototypes; 5-fold x 3 seeds)'); note(tab.to_string())
BEST = tab['overall'].idxmin(); note('BEST: %s (%.2f) vs reference %.2f' % (BEST, tab.loc[BEST, 'overall'], tab.loc['M0_ref', 'overall']))

In [ ]:
# Cell 5 - ZIP for the best variant (no VoxCeleb) + saved dev scores
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
fn_, vn_, lam = COMBOS[BEST]; F_raw, V_raw = build_train(fn_), build_train(vn_)
ep = int(np.median(R[R.combo == BEST]['best_ep']))
finals = [train_combo(F_raw, V_raw, np.arange(N_CH), None, seed=s, max_epochs=ep)[0] for s in SEEDS]
Fn_, Vn_, ch = dnorm(F_raw, V_raw, np.concatenate([np.arange(N_CH), EXT])); XFa = torch.tensor(Fn_, device=dev); XVa = torch.tensor(Vn_, device=dev)
w = 0.0
if lam > 0:
    Vt, Ft, _ = make_pairs(np.arange(N_CH), seed=777)
    w = np.mean([np.std(same_score(m, XFa[Ft], XVa[Vt])) for m in finals]) / (np.std(msm_score(Vt, Ft)) + 1e-9)
    note('fusion weight from TRAIN pairs only: lam %.1f x scale %.4f' % (lam, w))
sc = {}
for track in ['no_gender', 'gender']:
    for lang_ in LANG_FILE:
        split = track + '_' + lang_
        xf = torch.tensor(((build(fn_, split) - ch[0]) / ch[1]).astype(np.float32), device=dev); xv = torch.tensor(((build(vn_, split) - ch[2]) / ch[3]).astype(np.float32), device=dev)
        s_ = np.mean([same_score(m, xf, xv) for m in finals], axis=0)
        if lam > 0:
            s_ = s_ + lam * w * (-np.linalg.norm(MSMF[split] - MSMV[split], axis=1))
        sc[split] = s_
tag = BEST; OUT = '/content/sub_25_' + tag
for track in ['no_gender', 'gender']:
    os.makedirs(os.path.join(OUT, track), exist_ok=True)
    for lang_, fname in LANG_FILE.items():
        split = track + '_' + lang_; pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
        assert len(pair_ids) == len(sc[split]) and np.isfinite(sc[split]).all()
        with open(os.path.join(OUT, track, fname), 'w') as f:
            for pid, v in zip(pair_ids, -sc[split]):
                f.write('%s %.6f\n' % (pid, v))
zp = os.path.join(SUBDIR, '25_%s_noVox.zip' % tag)
with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
    for track in ['no_gender', 'gender']:
        for fname in LANG_FILE.values():
            zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
np.savez(os.path.join(RESDIR, 'devscores_25_%s.npz' % tag), **sc); note('Submission ZIP saved: ' + zp[len(BASE):])
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)